# AIML428 - Assignment1

## Steps Completed

All steps were completed.

## How to run

I make use of the GloVe embeddings. To set up the environment:

```
mkdir data
mv bbc-fulltext.zip data
cd data
wget https://nlp.stanford.edu/data/wordvecs/glove.2024.wikigiga.50d.zip
unzip glove.2024.wikigiga.50d.zip
```

To prepare the BBC CSV file:

```
convert_all.sh
```

It will create `bbc_converted.csv` in the current directory.

## Additional Packages

1. nltk

To prepare the report:




## Step 1 - Conversion

To prepare the BBC CSV file:

```
convert_all.sh
```

It will create `bbc_converted.csv` in the current directory.

## Step 2 - Term Frequency Classifier

In [20]:

import pandas as pd
from IPython.display import Markdown, display
from sklearn import tree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

In [21]:
def tableHeader():
    doc = "|Algorithm| label | accuracy | precision | recall | f1 |\n"
    doc += "| --- | --- | --- | --- |--- | --- |\n"
    return doc

def tableScore(labels, doc, prefix=None, y_true=None, y_prediction=None):
    rows = ""
    accuracy = accuracy_score(y_true, y_prediction)
    precisions = precision_score(y_true, y_prediction, labels=labels, average=None)
    recalls = recall_score(y_true, y_prediction, labels=labels, average=None)
    f1s = f1_score(y_true, y_prediction, labels=labels, average=None)
    for label, precision, recall, f1 in zip(labels, precisions, recalls, f1s):
        rows += f"| {prefix} | {label} | {accuracy:.4g} | {precision:.4g} | {recall:.4g} | {f1:.4g} |\n"
        prefix = " "
    return doc + rows

def tableFooter(doc):
    display(Markdown(doc))

In [22]:
df_csv = pd.read_csv('bbc_converted.csv')

feature_column = ['text']
target_column = 'category'

X_csv = df_csv[feature_column]
Y_csv = df_csv[target_column]
# unique converts to an NDArray, so the sort needs to happen first.
labels = Y_csv.sort_values().unique()
print(labels)

['business' 'entertainment' 'politics' 'sport' 'tech']


In [23]:
# https://xkcd.com/221/ - 4 is overused
random_seed = 221

### TermFrequency Classifiers

In [24]:
# Convert X_csv to TF.
tfidf_vectorizer=CountVectorizer()
# When converting from dataframe to scikit_learn, fit the columns, it expects an iterable.
X_tf = tfidf_vectorizer.fit_transform(X_csv.text)

# Split the data into Train and Test
X_tf_train, X_tf_test, y_tf_train, y_tf_test = train_test_split(
    X_tf, Y_csv, test_size= 0.3, random_state=random_seed)

doc = tableHeader()
# Now build a decision tree
tf_tree = tree.DecisionTreeClassifier(criterion="entropy", random_state=random_seed)
tf_tree = tf_tree.fit(X_tf_train, y_tf_train)

# Evaluate the tree using X_tf_train and y_tf_train
y_tf_train_prediction = tf_tree.predict(X_tf_train)
doc = tableScore(labels, doc, "TF DecisionTree Training", y_tf_train, y_tf_train_prediction)

# Now evaluate the tree using X_tf_test and y_tf_test
y_tf_prediction = tf_tree.predict(X_tf_test)
doc = tableScore(labels, doc,"TF DecisionTree Test", y_tf_test, y_tf_prediction)

# Now use Multinomial Naive Bayes - Multinomial works with text frequencies.
tf_mnb = MultinomialNB()
tf_mnb = tf_mnb.fit(X_tf_train, y_tf_train)

# Evaluate model using X_tf_train and y_tf_train
y_tf_train_prediction = tf_mnb.predict(X_tf_train)
doc = tableScore(labels, doc,"TF NaiveBayes Training", y_tf_train, y_tf_train_prediction)

# Now evaluate model using X_tf_test and y_tf_test
y_tf_prediction = tf_mnb.predict(X_tf_test)
doc = tableScore(labels, doc,"TF NaiveBayes Test", y_tf_test, y_tf_prediction)

# Now use Support Vector Machines.
tf_svm = SVC(random_state=random_seed)
tf_svm = tf_svm.fit(X_tf_train, y_tf_train)

# Evaluate model using X_tf_train and y_tf_train
y_tf_train_prediction = tf_svm.predict(X_tf_train)
doc = tableScore(labels, doc,"TF SVM Training", y_tf_train, y_tf_train_prediction)

# Now evaluate model using X_tf_test and y_tf_test
y_tf_prediction = tf_svm.predict(X_tf_test)
doc = tableScore(labels, doc,"TF SVM Test", y_tf_test, y_tf_prediction)

tableFooter(doc)


|Algorithm| label | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- |
| TF DecisionTree Training | business | 1 | 1 | 1 | 1 |
|   | entertainment | 1 | 1 | 1 | 1 |
|   | politics | 1 | 1 | 1 | 1 |
|   | sport | 1 | 1 | 1 | 1 |
|   | tech | 1 | 1 | 1 | 1 |
| TF DecisionTree Test | business | 0.7769 | 0.7216 | 0.8038 | 0.7605 |
|   | entertainment | 0.7769 | 0.7193 | 0.7257 | 0.7225 |
|   | politics | 0.7769 | 0.7521 | 0.7333 | 0.7426 |
|   | sport | 0.7769 | 0.8554 | 0.9281 | 0.8903 |
|   | tech | 0.7769 | 0.8421 | 0.6452 | 0.7306 |
| TF NaiveBayes Training | business | 0.9942 | 1 | 0.9858 | 0.9928 |
|   | entertainment | 0.9942 | 0.9927 | 0.9963 | 0.9945 |
|   | politics | 0.9942 | 0.9933 | 0.9966 | 0.995 |
|   | sport | 0.9942 | 1 | 1 | 1 |
|   | tech | 0.9942 | 0.9821 | 0.9928 | 0.9874 |
| TF NaiveBayes Test | business | 0.9716 | 0.9744 | 0.962 | 0.9682 |
|   | entertainment | 0.9716 | 1 | 0.9292 | 0.9633 |
|   | politics | 0.9716 | 0.937 | 0.9917 | 0.9636 |
|   | sport | 0.9716 | 1 | 0.9935 | 0.9967 |
|   | tech | 0.9716 | 0.9453 | 0.9758 | 0.9603 |
| TF SVM Training | business | 0.9788 | 0.9614 | 0.9915 | 0.9762 |
|   | entertainment | 0.9788 | 0.971 | 0.9817 | 0.9763 |
|   | politics | 0.9788 | 0.9859 | 0.9428 | 0.9639 |
|   | sport | 0.9788 | 0.9808 | 1 | 0.9903 |
|   | tech | 0.9788 | 1 | 0.9711 | 0.9853 |
| TF SVM Test | business | 0.9296 | 0.8851 | 0.9747 | 0.9277 |
|   | entertainment | 0.9296 | 0.9541 | 0.9204 | 0.9369 |
|   | politics | 0.9296 | 0.9444 | 0.85 | 0.8947 |
|   | sport | 0.9296 | 0.9615 | 0.9804 | 0.9709 |
|   | tech | 0.9296 | 0.9174 | 0.8952 | 0.9061 |


Naive Bayes performed best on the test, with a 97.16% accuracy.

## Step 3 - Term Frequency - Inverse Document Frequency Classifiers

In [25]:
# Convert X_csv to TF.
tfidf_vectorizer=TfidfVectorizer()
# When converting from dataframe to scikit_learn, fit the columns, it expects an iterable.
X_tfidf = tfidf_vectorizer.fit_transform(X_csv.text)

# Split the data into Train and Test
X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test = train_test_split(
    X_tfidf, Y_csv, test_size= 0.3, random_state=random_seed)

doc = tableHeader()

# Now build a decision tree
tfidf_tree = tree.DecisionTreeClassifier(criterion="entropy", random_state=random_seed)
tfidf_tree = tfidf_tree.fit(X_tfidf_train, y_tfidf_train)

# Evaluate the tree using X_tfidf_train and y_tfidf_train
y_tfidf_train_prediction = tfidf_tree.predict(X_tfidf_train)
train_accuracy = accuracy_score(y_tfidf_train, y_tfidf_train_prediction)
doc = tableScore(
    labels, doc, "TFIDF DecisionTree Training", y_tfidf_train, y_tfidf_train_prediction)

# Now evaluate the tree using X_tfidf_test and y_tfidf_test
y_tfidf_prediction = tfidf_tree.predict(X_tfidf_test)
doc = tableScore(labels, doc, "TFIDF DecisionTree Test", y_tfidf_test, y_tfidf_prediction)

# Now use Multinomial Naive Bayes - Multinomial works with text frequencies.
tfidf_mnb = MultinomialNB()
tfidf_mnb = tfidf_mnb.fit(X_tfidf_train, y_tfidf_train)

# Evaluate model using X_tfidf_train and y_tfidf_train
y_tfidf_train_prediction = tfidf_mnb.predict(X_tfidf_train)
doc = tableScore(
    labels, doc, "TFIDF NaiveBayes Training", y_tfidf_train, y_tfidf_train_prediction)

# Now evaluate model using X_tfidf_test and y_tfidf_test
y_tfidf_prediction = tfidf_mnb.predict(X_tfidf_test)
doc = tableScore(
    labels, doc, "TFIDF NaiveBayes Test", y_tfidf_test, y_tfidf_prediction)

# Now use Support Vector Machines.
tfidf_svm = SVC(random_state=random_seed)
tfidf_svm = tfidf_svm.fit(X_tfidf_train, y_tfidf_train)

# Evaluate model using X_tfidf_train and y_tfidf_train
y_tfidf_train_prediction = tfidf_svm.predict(X_tfidf_train)
doc = tableScore(
    labels, doc, "TFIDF SVM Training", y_tfidf_train, y_tfidf_train_prediction)

# Now evaluate model using X_tfidf_test and y_tfidf_test
y_tfidf_prediction = tfidf_svm.predict(X_tfidf_test)
doc = tableScore(
    labels, doc, "TFIDF SVM Test", y_tfidf_test, y_tfidf_prediction)

tableFooter(doc)


|Algorithm| label | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- |
| TFIDF DecisionTree Training | business | 1 | 1 | 1 | 1 |
|   | entertainment | 1 | 1 | 1 | 1 |
|   | politics | 1 | 1 | 1 | 1 |
|   | sport | 1 | 1 | 1 | 1 |
|   | tech | 1 | 1 | 1 | 1 |
| TFIDF DecisionTree Test | business | 0.7425 | 0.6894 | 0.7025 | 0.6959 |
|   | entertainment | 0.7425 | 0.675 | 0.7168 | 0.6953 |
|   | politics | 0.7425 | 0.6807 | 0.675 | 0.6778 |
|   | sport | 0.7425 | 0.9007 | 0.8889 | 0.8947 |
|   | tech | 0.7425 | 0.7436 | 0.7016 | 0.722 |
| TFIDF NaiveBayes Training | business | 0.9833 | 0.9831 | 0.9915 | 0.9873 |
|   | entertainment | 0.9833 | 0.9923 | 0.9451 | 0.9681 |
|   | politics | 0.9833 | 0.9704 | 0.9933 | 0.9817 |
|   | sport | 0.9833 | 0.9862 | 1 | 0.9931 |
|   | tech | 0.9833 | 0.9855 | 0.9783 | 0.9819 |
| TFIDF NaiveBayes Test | business | 0.9536 | 0.9563 | 0.9684 | 0.9623 |
|   | entertainment | 0.9536 | 1 | 0.8496 | 0.9187 |
|   | politics | 0.9536 | 0.8741 | 0.9833 | 0.9255 |
|   | sport | 0.9536 | 0.9682 | 0.9935 | 0.9806 |
|   | tech | 0.9536 | 0.9833 | 0.9516 | 0.9672 |
| TFIDF SVM Training | business | 0.9994 | 1 | 0.9972 | 0.9986 |
|   | entertainment | 0.9994 | 1 | 1 | 1 |
|   | politics | 0.9994 | 1 | 1 | 1 |
|   | sport | 0.9994 | 1 | 1 | 1 |
|   | tech | 0.9994 | 0.9964 | 1 | 0.9982 |
| TFIDF SVM Test | business | 0.9626 | 0.9118 | 0.981 | 0.9451 |
|   | entertainment | 0.9626 | 0.9818 | 0.9558 | 0.9686 |
|   | politics | 0.9626 | 0.9737 | 0.925 | 0.9487 |
|   | sport | 0.9626 | 0.9935 | 0.9935 | 0.9935 |
|   | tech | 0.9626 | 0.9669 | 0.9435 | 0.9551 |


## Report

There were a lot of datatype mismatches that required reading the model code to debug.

To properly evaluate the different models, K-cross validation should be used. The test set splits in the report should be consistent - the same random_seed was used. A proper evaluation would perform k-cross validation (with a fixed seed) and use that to compare model performance. That would avoid test/train splits being helpful to one model over another.

### Text Frequency

Naive Bayes performed best on Text Frequency, returning 97.16% accuracy. Decision Tree showed the most overfitting, with training accuracy at 100%, but a test accuracy of 77.69%.

Adding in the Inverse Document Frequency made most models perform worse. Only SVM performs better than the TF version, with an accuracy of 96.26%. Naive Bayes with TF was still the winner.

### TF-IDF

I believe that normalization works against the model for Decision Trees by compressing the available bits. However, SVM is better able to differentiate classes post IDF. This may be due to the vectors being more similar. Decision Trees are poor at classifying text. There are too many unique dimensions.

### CNN

The CNN in step 4 produced an accuracy of 89.52%, failing to beat TFIDF + SVM. I added Lemmatization, Oversampling and Early Stopping to improve performance. Of the oversampling algorithms, only Random was appropriate. The other available algorithms (Smote, ADASYN) required numeric dimensions. This is incorrect for documents, creating new vectors by interpolating between others would only create documents that don't make sense. Strangely I had a bug where I used "binary_crossentropy" as the loss function, but changing it to "categorical_crossentropy" had no effect on the final accuracy result.

### CNN with pretrained embedding

Writing the pretrained embedding was interesting. I learned that the official online documentation (https://keras.io/examples/nlp/pretrained_word_embeddings/) was broken, it breaks reading the GloVe file, and used a zip which wasn't required. This flowed through to other online examples. The example point of speech tag conversions from Treebank to WordNet were also buggy. Lemmatization would work against the pretrained embedding, so it was removed.

Switching to "categorical_crossentropy" resulted in poorer performance on the test set. The CNN with the pre-trained embedding had an accuracy of 93.71%. The use of a pre-trained embedding, along with the removal of lemmatization allows the embedding to find words that are similar. While the specific term in the document may not be present, a word close in the embedding space may be present, allowing the model to find the correct classification. The confusion matrix was interesting. It showed that there was some overlap between some of the classes, but not others. For example, Politics was frequently classified as Business, but nothing was misclassified as Sport.

### Bert

Bert took way too long to run, with 3 epochs taking 30 minutes. I tried to get it to run on my GPU, but I couldn't manage it in the time available.

The first run had an accuracy of 23.65%.

* 3 epochs - 23.65%
* 10 epochs - 23.65%
* 50 epochs - failed.

After a Jupyter server restart, the model started to fail. While investigating, I learned that the model picks settings like loss function based on the data provided. I am using OrdinalEncoder to translate the labels, which defaults to float64. However, float64 labels causes Bert to perform multi-label classification, using BCEWithLogitsLoss.

Converting to int64 and increasing the batch size to 32 resulted in the accuracy increasing to 48.05% after 3 epochs. The model also started to improve with each epoch. With a 5 epoch run, accuracy increased to 67.66%. This shows the model is converging, and with more epochs it should continue to improve.

* 3 epochs - 48.05%
* 5 epochs - 67.66%